In [0]:
%sql

CREATE TABLE dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details (
    -- Document identifiers
    document_id                VARCHAR(64) NOT NULL,
    document_cuid              VARCHAR(64),
    document_name              VARCHAR(512),
    folder_id                  VARCHAR(64),
    full_path                  VARCHAR(1024),

    -- Document metadata
    document_updated_ts        TIMESTAMP,
    document_scheduled         VARCHAR(64),
    document_created_by        VARCHAR(256),
    document_last_author       VARCHAR(256),

    -- Schedule identifiers
    schedule_id                VARCHAR(64),
    schedule_name              VARCHAR(512),

    -- Schedule instance metadata
    schedule_updated_ts        TIMESTAMP,
    schedule_format            VARCHAR(64),
    schedule_status            VARCHAR(64),

    -- ✅ Key motivation fields
    repetition_type            VARCHAR(32),  -- once | hourly | daily | weekly | monthly | calendar
    repetition                 STRING,          -- structured repetition block

    destination_type           VARCHAR(32),  -- mail | inbox | file | ftp
    destination                STRING,          -- structured destination block

    parameters                 STRING,          -- schedule parameters
    Instance_json              STRING,          -- schedule full json API respons
    -- Extraction / execution stats  
    ingestion_ts               TIMESTAMP
    
    );




Below to use create the table for Schedule reports scan output, only the ones has schedule and with a record in the instance got captured. 

In [0]:
%sql
create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS
select distinct EX.document_id, EX.document_cuid, EX.document_name, EX.full_path as BO_folder_path, EX.document_created_by as owner, EX.schedule_updated_ts as last_scheduled_execTS, EX.schedule_format as output_format, (case when upper(EXE.destination_type)="MAIL" then true else false end) as email_delivery, (case when upper(EXE.destination_type)="FILESYSTEM" then true else false end) as file_delivery, (case when UPPER(EXA.schedule_activeness)="ACTIVE" and upper(EXA.repetition_type) in ("MONTHLY","DAILY","WEEKLY","HOURLY") then true else false end ) as Active_Schedule from  dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details as EX 
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_enriched as EXE
on EX.document_id=EXE.document_id and EX.schedule_id=EXE.schedule_id
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_active as EXA
on EX.document_id=EXA.document_id and EX.schedule_id=EXA.schedule_id;


In [0]:
%sql
    
    
create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted_email AS
WITH emails AS (
  SELECT
    TRIM(email) AS email,
    document_id,
    schedule_id,
    document_name,
    destination_type
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_enriched
  LATERAL VIEW explode(
    CASE
      WHEN lower(destination_type) = 'mail'
        THEN split(regexp_replace(sent_to, '[;,]+', ','), ',')
      ELSE array()
    END
  ) exploded_email AS email
  WHERE lower(destination_type) = 'mail'
    AND TRIM(email) != ''
    AND instr(email, '@') > 0
),
linage_sources AS (
  SELECT 
    upper(TRIM(Document_Id)) AS Document_Id,
    ARRAY_JOIN(ARRAY_SORT(COLLECT_SET(TRIM(regexp_replace(MI_SOURCE, '\\(.*?\\)|\\s+(TABLE|VIEW)\\s*$', '')))), ', ') AS MI_SOURCE
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage
  WHERE MI_SOURCE IS NOT NULL AND upper(trim(MI_SOURCE)) NOT IN ('NOT AVAILABLE')
  GROUP BY upper(TRIM(Document_Id))
)
SELECT DISTINCT
  e.email,
  e.document_id,
  e.schedule_id,
  e.document_name,
  e.destination_type,
  l.MI_SOURCE,
  current_date() as ingestion_date
FROM emails e
LEFT JOIN linage_sources l
  ON e.document_id = l.Document_Id
ORDER BY email ASC

In [0]:
%sql
    
    
    

create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted_sharedlocation AS
WITH file_locations AS (
  SELECT
    DISTINCT
    upper(TRIM(file_location)) AS file_location,
    upper(TRIM(document_id)) AS document_id,
    upper(TRIM(schedule_id)) AS schedule_id,
    upper(TRIM(document_name)) AS document_name,
    upper(TRIM(destination_type)) AS destination_type
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_enriched
  LATERAL VIEW explode(
    CASE
      WHEN lower(destination_type) = 'filesystem'
        THEN split(regexp_replace(file_path, '[;,]+', ','), ',')
      ELSE array()
    END
  ) exploded_fileloc AS file_location
  WHERE lower(destination_type) = 'filesystem'
    AND upper(TRIM(file_location)) != ''
),
linage_sources AS (
  SELECT 
    upper(TRIM(Document_Id)) AS Document_Id,
    ARRAY_JOIN(ARRAY_SORT(COLLECT_SET(TRIM(regexp_replace(MI_SOURCE, '\\(.*?\\)|\\s+(TABLE|VIEW)\\s*$', '')))), ', ') AS MI_SOURCE
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage
  WHERE MI_SOURCE IS NOT NULL and upper(trim(MI_SOURCE)) not in ('NOT AVAILABLE')
  GROUP BY upper(TRIM(Document_Id))
)
SELECT DISTINCT
  f.file_location,
  f.document_id,
  f.schedule_id,
  f.document_name,
  f.destination_type,
  l.MI_SOURCE,
  current_date() as ingestion_date
FROM file_locations f
LEFT JOIN linage_sources l
  ON f.document_id = l.Document_Id
ORDER BY file_location ASC;

-- SELECT file_location, count(distinct document_id) as cnt 
-- FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted_sharedlocation
-- GROUP BY file_location
-- ORDER BY cnt DESC

-- select * FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted_sharedlocation

In [0]:
%sql


select distinct CMS.Report_ID
from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as WAA
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_cms_file_metadata as CMS
on WAA.WEBI_Doc_ID = CMS.Report_ID
where CMS.Report_ID is not null
and CMS.scheduled=true
order by CMS.Report_ID desc;

select count(*) from _sqldf

In [0]:
%sql

select count(distinct Document_Id) from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata;

select distinct FID.Report_ID  
from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.bo_user_access_audit as WEBA
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.sapbo_full_doc_id as FID
on upper(FID.Report_CUID) =upper(WEBA.Object_ID)
where WEBA.Object_ID is not null;

CREATE OR REPLACE TABLE dataplatform01_central_dev_catalog_01.custom_sap_bo.Webi_schedule_scan_tracker
select distinct FID.Report_ID, 
(Case when WEBA.Object_ID is not null then "Active" end) as Audit, 
CMS.scheduled as CMS_scheduled,
"" as Scanned,
"" as Scanned_Date,
"" as Result
from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.sapbo_full_doc_id as FID
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.bo_user_access_audit as WEBA
on upper(FID.Report_CUID) =upper(WEBA.Object_ID)
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as CMS
on upper(FID.Report_ID) = upper(CMS.Document_Id)
where upper(FID.Report_Type)="WEBI"
order by FID.Report_ID asc;


In [0]:
%sql
select distinct destination_type, destination from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details
where destination_type is null;

select distinct repetition_type, repetition from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details
where repetition_type is null;

select file_location, 
-- MI_SOURCE, 
count(distinct document_id) from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted_sharedlocation
group by file_location
-- , MI_SOURCE
order by count(distinct document_id) desc
;
select MI_SOURCE, count(distinct document_id) from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted_email
group by MI_SOURCE
order by count(distinct document_id) desc

In [0]:
%sql
select 
  destination_type, 
  count(distinct document_id) as cnt,
  concat(format_number(100.0 * count(distinct document_id) / sum(count(distinct document_id)) over (), 1), '%') as pct_of_total
from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details
where  schedule_status="Completed"
group by  destination_type
order by cnt desc
;

select destination_type, destination from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details
where schedule_status="Completed"
and  destination_type ="filesystem"
order by

In [0]:
%sql
  SELECT
   repetition_type, count(distinct document_id) as cnt
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_active
  WHERE schedule_activeness = 'Active'
  group by repetition_type
  order by cnt desc

In [0]:
%sql
    
select 
  UP.BU, 
  count(distinct RC.email) as Email_recepients,
  count(distinct case when wa.username is not null then RC.email end) as BO_active_users
from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted_email as RC
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile as UP 
  on upper(TRIM(RC.email))=upper(trim(UP.EmailAddress))
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as wa
  on upper(trim(wa.UserName))=upper(trim(UP.UserName))

group by UP.BU

union all

select 
  'TOTAL emails' as BU, 
  count(distinct RC.email) as Email_recepients,
  count(distinct case when wa.username is not null then RC.email end) as BO_active_users
from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted_email as RC
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile as UP 
  on upper(TRIM(RC.email))=upper(trim(UP.EmailAddress))
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as wa
  on upper(trim(wa.UserName))=upper(trim(UP.UserName))

order by Email_recepients desc
;

SELECT * EXCEPT(sk) FROM (
  select 
    "Shared location " as Category, 
    RC.file_location as sent_to, 
    count(distinct RC.document_id) as report_total_cnt, 
    count(distinct case when wa.username is not null then RC.document_id end) as report_active_cnt,
    case when count(distinct case when wa.username is not null then RC.document_id end) > count(distinct RC.document_id) then 'WARNING' else '' end as warning_flag,
    3 as sk
  from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted_sharedlocation as RC
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as wa
  on upper(trim(RC.document_id))=upper(trim(wa.WEBI_Doc_ID))
  group by Category, file_location

  union all

  select 
    "Sent to email" as Category, 
    RC.email as sent_to, 
    count(distinct RC.document_id) as report_total_cnt, 
    count(distinct case when wa.username is not null then RC.document_id end) as report_active_cnt,
    case when count(distinct case when wa.username is not null then RC.document_id end) > count(distinct RC.document_id) then 'WARNING' else '' end as warning_flag,
    4 as sk
  from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted_email as RC
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as wa
  on upper(trim(RC.document_id))=upper(trim(wa.WEBI_Doc_ID))
  group by Category, email

  union all

  select
    "Grand Total" as Category,
    "" as sent_to,
    count(distinct document_id) as report_total_cnt,
    count(distinct case when wa.username is not null then document_id end) as report_active_cnt,
    "" as warning_flag,
    1 as sk
  from (
    select document_id from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted_sharedlocation
    union
    select document_id from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted_email
  ) as all_docs
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as wa
  on upper(trim(all_docs.document_id))=upper(trim(wa.WEBI_Doc_ID))

  union all

  select
    "TOTAL emails" as Category,
    "" as sent_to,
    count(distinct RC.document_id) as report_total_cnt,
    count(distinct case when wa.username is not null then RC.document_id end) as report_active_cnt,
    "" as warning_flag,
    2 as sk
  from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted_email as RC
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as wa
  on upper(trim(RC.document_id))=upper(trim(wa.WEBI_Doc_ID))
) t
ORDER BY sk ASC, report_total_cnt DESC;

In [0]:
%sql
WITH base AS (
  SELECT 
    repetition_type,
    date_format(date_trunc('month', schedule_updated_ts), 'yyyy-MM') as exec_month,
    document_id
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_active
  WHERE schedule_activeness = 'Active'
    AND schedule_updated_ts IS NOT NULL
    AND repetition_type != 'once'
    AND schedule_updated_ts >= '2025-03-01'
),
pivoted AS (
  SELECT * FROM base
  PIVOT (
    count(distinct document_id)
    FOR exec_month IN (
      '2026-04' as `2026-04`,
      '2026-03' as `2026-03`,
      '2026-02' as `2026-02`,
      '2026-01' as `2026-01`,
      '2025-12' as `2025-12`,
      '2025-11' as `2025-11`,
      '2025-10' as `2025-10`,
      '2025-09' as `2025-09`,
      '2025-08' as `2025-08`,
      '2025-07' as `2025-07`,
      '2025-06' as `2025-06`,
      '2025-05' as `2025-05`,
      '2025-04' as `2025-04`,
      '2025-03' as `2025-03`
    )
  )
),
grand_total AS (
  SELECT
    count(distinct document_id) as gt
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_active
  WHERE schedule_activeness = 'Active'
)
SELECT * EXCEPT(sk) FROM (
  -- Data rows
  SELECT repetition_type, 1 as sk,
    cast(coalesce(`2026-04`,0) as string) as `2026-04`,
    cast(coalesce(`2026-03`,0) as string) as `2026-03`,
    cast(coalesce(`2026-02`,0) as string) as `2026-02`,
    cast(coalesce(`2026-01`,0) as string) as `2026-01`,
    cast(coalesce(`2025-12`,0) as string) as `2025-12`,
    cast(coalesce(`2025-11`,0) as string) as `2025-11`,
    cast(coalesce(`2025-10`,0) as string) as `2025-10`,
    cast(coalesce(`2025-09`,0) as string) as `2025-09`,
    cast(coalesce(`2025-08`,0) as string) as `2025-08`,
    cast(coalesce(`2025-07`,0) as string) as `2025-07`,
    cast(coalesce(`2025-06`,0) as string) as `2025-06`,
    cast(coalesce(`2025-05`,0) as string) as `2025-05`,
    cast(coalesce(`2025-04`,0) as string) as `2025-04`,
    cast(coalesce(`2025-03`,0) as string) as `2025-03`
  FROM pivoted

  UNION ALL

  -- Total row
  SELECT 'TOTAL', 2,
    cast(sum(coalesce(`2026-04`,0)) as string),
    cast(sum(coalesce(`2026-03`,0)) as string),
    cast(sum(coalesce(`2026-02`,0)) as string),
    cast(sum(coalesce(`2026-01`,0)) as string),
    cast(sum(coalesce(`2025-12`,0)) as string),
    cast(sum(coalesce(`2025-11`,0)) as string),
    cast(sum(coalesce(`2025-10`,0)) as string),
    cast(sum(coalesce(`2025-09`,0)) as string),
    cast(sum(coalesce(`2025-08`,0)) as string),
    cast(sum(coalesce(`2025-07`,0)) as string),
    cast(sum(coalesce(`2025-06`,0)) as string),
    cast(sum(coalesce(`2025-05`,0)) as string),
    cast(sum(coalesce(`2025-04`,0)) as string),
    cast(sum(coalesce(`2025-03`,0)) as string)
  FROM pivoted

  UNION ALL

  -- Percentage row
  SELECT '% of Total', 3,
    concat(format_number(100.0 * coalesce(sum(`2026-04`),0) / gt, 1), '%'),
    concat(format_number(100.0 * coalesce(sum(`2026-03`),0) / gt, 1), '%'),
    concat(format_number(100.0 * coalesce(sum(`2026-02`),0) / gt, 1), '%'),
    concat(format_number(100.0 * coalesce(sum(`2026-01`),0) / gt, 1), '%'),
    concat(format_number(100.0 * coalesce(sum(`2025-12`),0) / gt, 1), '%'),
    concat(format_number(100.0 * coalesce(sum(`2025-11`),0) / gt, 1), '%'),
    concat(format_number(100.0 * coalesce(sum(`2025-10`),0) / gt, 1), '%'),
    concat(format_number(100.0 * coalesce(sum(`2025-09`),0) / gt, 1), '%'),
    concat(format_number(100.0 * coalesce(sum(`2025-08`),0) / gt, 1), '%'),
    concat(format_number(100.0 * coalesce(sum(`2025-07`),0) / gt, 1), '%'),
    concat(format_number(100.0 * coalesce(sum(`2025-06`),0) / gt, 1), '%'),
    concat(format_number(100.0 * coalesce(sum(`2025-05`),0) / gt, 1), '%'),
    concat(format_number(100.0 * coalesce(sum(`2025-04`),0) / gt, 1), '%'),
    concat(format_number(100.0 * coalesce(sum(`2025-03`),0) / gt, 1), '%')
  FROM pivoted CROSS JOIN grand_total
  GROUP BY gt
) t
ORDER BY sk, repetition_type

In [0]:
%sql
select distinct file_path from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_enriched

In [0]:
%sql
select repetition_type, count(distinct case when repetition_enddate is null then  document_id end ) as no_schedule_cnt, count(distinct case when schedule_activeness = 'Active' then document_id end) as active_cnt, count (distinct case when schedule_activeness = 'Inactive' and repetition_enddate is not null then document_id end) as inactive_cnt, count(distinct document_id) as total_cnt from  dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_active
group by repetition_type


In [0]:
%sql
select * from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_enriched



In [0]:
%sql
select * from 
-- dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted_email
dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted_sharedlocation